In [8]:
import numpy as np
import torch
import torch.nn as nn
import timm
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

# Class 3: Advanced 2D Visual Representation

本节课围绕三个关键节点展开：**ViT**、**DINOv2**、**MAE**。  
它们共同回答的不是“怎样做一个分类器”，而是一个更底层的问题：

> **在视觉任务高度碎片化的情况下，我们能不能学到一种通用、可迁移、可被不同任务读取的视觉表征？**

所以这节课可以分成两半：

- **ViT**：解决形式问题——如何把图像变成 Transformer 可以处理的 token 序列。
- **DINOv2 / MAE**：解决表征问题——这些 token 到底应该承载什么样的视觉世界结构。


## Recap & Thinking

- **CNN的不足**：传统 CNN 通过堆叠卷积层建立感受野，全局建模能力弱，且归纳偏置（inductive bias）强，迁移到复杂任务时往往需要专门设计任务头。
- **LLM怎么做的**：Transformer 的 Self-Attention 机制让每个 token 直接关注所有其他 token，一步建立全局依赖；更重要的是，很多 NLP 任务都可以被统一成“输入 token → 输出 token”。
- **CV真正难在哪里**：视觉任务的目标空间很难统一。分类输出类别，检测输出框，分割输出像素掩码，深度估计输出连续几何量，3D/位姿任务还牵涉到空间坐标系。  
  因此，视觉基础模型很难只靠一个类似 next-token prediction 的目标吃遍所有任务。
- **本节课的主线**：既然输出目标难统一，那就先追求一个更现实的统一对象——**通用视觉表征**。它不直接替所有任务做最终答案，而是提供一个足够好的中间层，让不同任务都能从中读取自己需要的信息。


## LLM vs. CV：序列化图像的关键挑战

我们从LLM和CV任务的本质区别出发

<div align="center">

| | NLP (LLM) | CV (图像) |
|---|---|---|
| 基本单元 | Token（词/子词） | Pixel（像素） |
| 序列长度 | 200K | 单图大约 224×224 = 50,176（太长！）|
| Masked Attention | 有语序，可 causal mask（节省显存） | 无天然顺序，不可 mask |
| Scaling | 序列长度灵活 | 分辨率×2 → token 数×4，算力骤增 |
</div>

**解决方案（ViT的核心思路）**：将图像切分为固定大小的 **Patch**（图块），每个 Patch 作为一个 token，大幅压缩序列长度。

<div align="center">
  <img src="imgs/ViT.png" alt="ViT 模型结构" width="800">
</div>

---
## Part 1: Vision Transformer (ViT)

> 论文：*An Image is Worth 16x16 Words* (Dosovitskiy et al., 2020)

### 核心流程
1. 将图像切分为 `P×P` 的 Patch（如 16×16），共 `N = (H/P)×(W/P)` 个
2. 每个 Patch 展平后经线性投影映射为 D 维 embedding
3. 拼接一个可学习的 `[CLS]` token（用于分类）
4. 加上可学习的 **Position Embedding**
5. 送入标准 Transformer Encoder（多层 Multi-Head Self-Attention + FFN）
6. 取 `[CLS]` token 的输出接分类头

### 从公式到代码：图像怎样变成一串 token？

有了前面的直观图示，现在可以真正“落地”到代码上：一张 224×224 的彩色图像，会被我们切成一块块 16×16 的小方块，每一块都被看成一句话里的一个“词”。

`PatchEmbedding` 这段实现做的事情，其实就两件：
- 用一个步长等于核大小的卷积，把整张图按网格切开，每个格子压成一个向量；
- 把这些向量排成一行，得到长度为 N 的序列 `(B, N, D)`，后面就可以直接送进 Transformer。

In [5]:
# ── 1. Patch Embedding ──────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    """
    将图像切分成 Patch 并投影到 embed_dim 维空间。
    等价于一个 kernel_size=patch_size, stride=patch_size 的卷积。
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        # 2D Conv 实现 patch 切分 + 线性投影
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, x):          # x: (B, C, H, W)
        x = self.proj(x)           # -> (B, embed_dim, H/P, W/P)
        x = x.flatten(2)           # -> (B, embed_dim, N)
        x = x.transpose(1, 2)     # -> (B, N, embed_dim)
        return x

patch_embed = PatchEmbedding(img_size=224, patch_size=16, embed_dim=768)
dummy_img = torch.randn(2, 3, 224, 224)   # batch=2
out = patch_embed(dummy_img)
print(f"输入图像: {dummy_img.shape}")
print(f"Patch数量: {patch_embed.num_patches}  (14×14)")
print("压缩比例:", round(patch_embed.num_patches / (224 * 224) * 100, 2), "%")
print(f"Patch Embedding输出: {out.shape}  (B, N, D)")

输入图像: torch.Size([2, 3, 224, 224])
Patch数量: 196  (14×14)
压缩比例: 0.39 %
Patch Embedding输出: torch.Size([2, 196, 768])  (B, N, D)


### 把积木搭成一个完整的 ViT 小模型

当我们知道如何把图像切成 patch token 之后，接下来就是把这些“积木”拼起来：在 `MiniViT` 里，你可以看到课程开头提到的那条完整路线——patch 序列进来，前面加上一个代表整张图的 `[CLS]`，再配上位置编码，最后丢进一堆 Transformer Encoder 层里。

这一小节的代码可以当成“放大版简笔画”，和论文中的框图一一对应：每一行都能在图里找到自己的角色。

In [9]:
# ── 2. Mini-ViT ─────────────────────────────────
class MiniViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=1000,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2

        # Patch Embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)

        # 可学习的 [CLS] token 和 Position Embedding
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim)) # （仅作演示，实际使用一般是RoPE等）

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # 分类头
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)                              # (B, N, D)
        cls = self.cls_token.expand(B, -1, -1)              # (B, 1, D)
        x = torch.cat([cls, x], dim=1)                      # (B, N+1, D)
        x = x + self.pos_embed                              # 加位置编码
        x = self.transformer(x)                             # (B, N+1, D)
        x = self.norm(x[:, 0])                              # 取 [CLS] token
        return self.head(x)                                  # (B, num_classes)


mini_vit = MiniViT(img_size=224, patch_size=16, num_classes=10,
                   embed_dim=192, depth=4, num_heads=3)
dummy = torch.randn(1, 3, 224, 224)
logits = mini_vit(dummy)
print(f"模型参数量: {sum(p.numel() for p in mini_vit.parameters())/1e6:.1f}M")
print(f"输出 logits shape: {logits.shape}")

模型参数量: 2.0M
输出 logits shape: torch.Size([1, 10])


### 看一眼”工业界“用的 ViT 长什么样

前面我们自己手写了一个toy ViT，现在换一个视角：如果是在实际工程项目里，大家几乎不会从零开始搭网络，而是直接调用类似 `timm` 这样的模型库。

这段代码做的，就是把论文中的 ViT-B/16 以“预训练模型”的形式拿过来，顺手瞄一眼它的关键超参数，然后抽出一条 768 维的 `[CLS]` 特征。它和我们刚刚实现的 `MiniViT` 在结构上高度对应，只是规模更大、训练更充分，是后面 DINOv2、MAE 这些表征学习方法的共同底座。


In [13]:
# ── 3. 加载预训练 ViT（使用 timm 库）──────────────────────────────────
# 加载在 ImageNet-21k 上预训练的 ViT-B/16
# pretrained=True 会自动下载权重（~330MB），首次运行需要网络
model = timm.create_model('vit_base_patch16_224', pretrained=True)
model.eval()

# 查看模型结构关键参数
print(f"Patch Size:   {model.patch_embed.patch_size}")
print(f"Embed Dim:    {model.embed_dim}")
print(f"Depth:        {len(model.blocks)}")
print(f"Num Heads:    {model.blocks[0].attn.num_heads}")
print(f"参数总量:      {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# 前向推理，提取 [CLS] 特征
dummy = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    features = model.forward_features(dummy)  # (1, 197, 768)
    cls_feat  = features[:, 0, :]             # [CLS] token -> (1, 768)
print(f"\n[CLS] 特征维度: {cls_feat.shape}")
print("=> 这个 768 维向量可直接用于下游任务（分类、检索等）")

Patch Size:   (16, 16)
Embed Dim:    768
Depth:        12
Num Heads:    12
参数总量:      86.6M

[CLS] 特征维度: torch.Size([1, 768])
=> 这个 768 维向量可直接用于下游任务（分类、检索等）


### ViT 就是“版本答案”了吗？
至此，看起来我们已经解决了一个“心腹大患”。我们已经成功的把 LLM 任务里大杀四方的框架套用在了 CV 领域。是不是可以无脑迁移一波，等着收获胜利即可？

### NO!
> 2024 ICLR：*Vision Transformers Need Registers* (Darcet et al.)
<div align="center">
  <img src="imgs/register_token.png" alt="Register Token" width="800">
</div>

这套框架至今都活跃在一线的视觉大模型里，例如：
> 2023 ICCV Oral：*Scalable Diffusion Models with Transformers* (Xie et al.)
<div align="center">
  <img src="imgs/DiT.png" alt="DiT 模型结构" width="800">
</div>

> 2025 CVPR Best Paper：*VGGT: Visual Geometry Grounded Transformer* (Wang et al.)
<div align="center">
  <img src="imgs/VGGT.png" alt="VGGT 模型结构" width="800">
</div>

> 2025 ICCV：*SpatialTrackerV2: 3D Point Tracking Made Easy* (Xiao et al.)
<div align="center">
  <img src="imgs/spatialtracker.png" alt="VGGT 模型结构" width="800">
</div>

## LLM vs. CV **AGAIN**

<div align="center">
<video width="600" controls muted>
    <source src="imgs/will.mp4" type="video/mp4">
</video>
</div>

ViT 的 Patchify 已经在「序列化」上弥合了图像与文本的gap，但为何 LLM 早已能长篇推理，而视觉/视频大模型仍易出现模糊与鬼影？根本矛盾不在架构本身，而在 **Token 空间的物理与数学本质**。

- **物理层面**：语言是人类文明的**离散符号**，视觉是自然界的**连续高维物理量**（$H\times W\times C$ 上的连续矩阵）。
- **数学层面**：将连续视觉特征当作「连续 Token」塞进为离散符号设计的自回归框架，会触发**联合概率密度不可解**、**维数灾难**与 **MSE 的回归均值/模糊**等问题，形成训练上的「泥潭」。

### 为什么连续的 Token 空间是训练的"泥潭"？

在 ViT 与 LLM 的对比中，核心矛盾在于 **离散符号（Discrete Symbols）与连续信号（Continuous Signals）** 的本征差异。下面从信息论与拓扑、损失与概率、注意力与流形几何几方面说明：为什么连续 Token 是训练的"泥潭"。

#### 0. 信息论与拓扑鸿沟：信息密度不对等与流形假说悖论

**信息密度极度不对等**：离散文本 Token 是经人类认知高度压缩的语义符号，单位自信息量大；连续视觉 Patch 是物理世界未经压缩的原始采样，空间平滑与冗余强。

让我们在脑海里做一组小实验：
- step 1: 想一想“猫咪”这个词，你认为你和其他人能够对齐吗？
- step 2: 在脑中渲染一个"猫咪"，你认为你和其他人还能够对齐吗？

#### 1. 损失函数的"锐度"与梯度动力学

LLM 在离散空间（词表 $V$）上通过 **交叉熵（Cross-Entropy）** 优化，CV 在连续空间上通常用 **均方误差（MSE）**。

**离散（LLM）：** $L_{LLM} = -\log \bigl( e^{z_{target}} \big/ \sum_{j=1}^{V} e^{z_j} \bigr)$。$e^z$ 的指数性使梯度 $\frac{\partial L}{\partial z}$ 在预测偏差时呈指数陡峭，形成"非黑即白"的反馈与明确决策边界。

**连续（CV）：** $L_{CV} = \| \hat{x} - x \|_2^2$，导数为 $2(\hat{x}-x)$，梯度平滑。**关键**：最小化 MSE 的最优解是条件期望 $\hat{Y}=\mathbb{E}[Y|X]$。若真实分布为**多峰**（一 Token 等可能为「垂直锐边」或「水平锐边」），模型会被迫输出二者的**算术平均**——在像素/特征空间即一团灰蒙蒙、无高频结构的模糊块（**回归均值**）。这是连续 Token + MSE 在生成保真度与纹理细节上长期瓶颈的数学根源。

**FastAvatar的训练经验：** Mask Loss设置不当导致渲染画面偏灰。

#### 2. 注意力机制的"秩"坍缩与距离集中

Transformer 的核心是 $A=\mathrm{softmax}(QK^\top)$，依赖 Token 间区分度。

**离散 Token**：Embedding 形成近似正交的聚类，$X$ 常满秩，$A$ 对比度高。

**连续 Token**：Patch 空间自相关强，$X$ 低秩，$\mathrm{Cov}(X_{patch})$ 相关性高，注意力被"摊薄"，$A$ 趋向各向同性。

高维连续空间中还存在**距离度量崩溃**（测度集中）：任意两随机向量间欧氏/余弦距离趋于常数，所有点仿佛落在高维超球壳上等距，导致注意力无法区分关键语义 Token 与冗余背景，退化为**注意力扩散**（Attention Diffusion）与近似全局平均，丧失细粒度空间与结构信息。


#### 3. 下游任务空间的本征碎裂：为什么视觉无法像 LLM 一样「一招鲜吃遍天」

前述讨论集中在**生成任务**上（联合概率、MSE 回归均值等）。若从**下游任务**的形态看，LLM 与 CV 的差异同样深刻：LLM 能靠同一套「Next Token Prediction」统一几乎所有 NLP 任务，而 CV 的下游任务在数学定义与拓扑空间上高度异构，难以收敛为单一训练目标。

**3.1 NLP 的「一招鲜」：Text-to-Text 与统一损失**

大语言模型之所以「一招鲜吃遍天」，是因为人类语言的下游任务（情感分类、机器翻译、代码生成、逻辑问答等）在数学上都可以**无损地收敛为「文本到文本（Text-to-Text）」**的序列生成：输入是离散 Token，输出永远是从同一词表采样的离散 Token 分布。模型自始至终只需优化**同一个损失**——分类交叉熵（Categorical Cross-Entropy）。这种架构与目标的绝对统一，使梯度方向一致，能通过 Scaling 实现多能力同步涌现。

**3.2 计算机视觉任务的物理与几何异构性**

计算机视觉的目标是解析**三维物理世界在二维平面的投影**，下游任务不仅在颗粒度上五花八门，而且**输出空间跨越完全不同的拓扑与量纲**：

| 任务 | 输出形态 | 数学空间 / 损失 |
|------|----------|------------------|
| 图像分类 | 全局离散类别 | 低维离散分布，交叉熵（与 NLP 最接近）|
| 目标检测 | 边界框 $[x, y, w, h]$ | 二维连续坐标，需 IoU / GIoU / CFIoU 等几何感知损失 |
| 语义分割 | 密集像素级类别掩码 | 高分辨率离散图，逐像素交叉熵 |
| 深度估计 | 每像素 Z 轴距离（2.5D）| 连续回归，存在尺度模糊（scale ambiguity）|
| 3D 重建 / 6D 位姿 | 体素/点云；$R \in SO(3)$, $T \in \mathbb{R}^3$ | 连续几何空间，需特殊参数化与损失 |

这种**任务目标的物理异构性**要求 CV 模型输出不同形态的连续/离散变量；若强行用「连续 Token + 单一回归损失」统一建模，会在数学与优化上遭遇下述困境。

**3.3 典型视觉任务中连续回归的数学困境**

- **深度估计**：单目 2D→3D 是病态逆问题，存在严重尺度模糊。早期纯 MSE 连续回归收敛慢、易陷局部最优。SOTA 方法（如 DORN、AdaBins）采用**间距递增离散化（SID）**，将连续深度切成离散区间（Bins），把连续回归转化为有序分类，反向说明直接连续深度回归的失败。
- **目标检测**：直接对边界框四坐标做 MSE 缺乏平移不变性、对异常值敏感，且无法反映框与框的真实重叠度。因此需要 IoU、GIoU、CFIoU 等**几何代理损失**来约束连续坐标预测。
- **3D 重建**：让 Transformer 直接输出连续体素/点云会遭遇维数灾难与显存爆炸。占位网络（Occupancy Networks）、神经辐射场（NeRF）等将 3D 表面表示为**隐式场**（连续空间中的决策边界或密度场），本质仍是借助分类或隐式函数来规避「直接高维连续坐标回归」的不稳定性。

**3.4 多任务学习的梯度冲突（Task Interference）**

当试图构建「视觉通用基础模型」——例如在同一 Transformer 上同时优化深度回归的 L1、检测的 IoU、分割的 Cross-Entropy——来自**异构损失**的梯度会在共享参数空间中发生**方向冲突**（Negative Transfer）。底层特征表达能力被不同任务互相抵消，模型易陷入「互相遗忘」。这种从物理世界到损失函数的**任务空间碎片化**，是 CV 难以像 LLM 那样仅靠「预测下一个视觉 Token」就统一所有下游任务的根本原因之一。

**小结**：在**生成**上，连续 Token 受制于配分函数不可解与 MSE 回归均值；在**下游任务**上，CV 的任务形态从 2D 分类到 6D 位姿、从离散掩码到连续几何，无法像 NLP 一样统一为单一的 Text-to-Text + 交叉熵，从而难以实现真正的「一招鲜吃遍天」。

---
## Part 2: DINOv2 — 通用视觉表征的雏形

> 论文：*DINOv2: Learning Robust Visual Features without Supervision* (Oquab et al., 2023)  
> 链接：<https://arxiv.org/abs/2304.07193>

前面我们已经讲过：CV 和 LLM 最大的差别之一，不只是输入模态不同，而是**任务定义方式不同**。

LLM 的很多能力可以收敛到同一个形式：给一串 token，预测下一串 token。  
但 CV 不一样：

| 任务 | 输出是什么 | 为什么难统一 |
|---|---|---|
| 分类 | 一个类别分布 | 全局语义 |
| 检测 | 多个连续边界框 | 几何位置 + 类别 |
| 分割 | 每个像素/patch 的类别 | 密集空间结构 |
| 深度估计 | 每个位置的连续深度 | 3D 几何与尺度 |
| 对应 / 跟踪 | 跨图像或跨帧的点匹配 | 局部语义 + 几何一致性 |

所以 DINOv2 的宏观意义，不是“没有标签也能训练”这么简单，而是在回答一个更关键的问题：

> **如果 CV 的最终任务很难统一，我们能否先统一中间表征？**

换句话说，DINOv2 想学到一种视觉世界的“通用坐标系”：一张图里的每个 patch，不只是一个像素块，而是落在某个语义-几何空间里的点。下游任务不必重新从像素开始理解世界，而是直接读取这个坐标系。


### DINOv2 想让 patch token 承载什么？

ViT 已经把图像切成 patch token，但这只是形式上的 token 化。真正的问题是：

> **一个好的视觉 patch token，应该保存什么信息？**

从通用表征的角度看，DINOv2 希望 patch feature 同时具备几种性质：

| 表征性质 | 直觉解释 | 对下游任务的意义 |
|---|---|---|
| **对象性** | 同一物体内部的 patch feature 更接近，边界处发生明显变化 | 分割、检测、前景提取 |
| **语义一致性** | 不同图片中相似部件/类别的 feature 靠近 | 检索、少样本分类、开放类别迁移 |
| **密集可读性** | 每个 patch 都有有用表示，而不只是 `[CLS]` 有用 | 分割、深度、对应、跟踪 |
| **跨域鲁棒性** | 换数据集、换风格、换任务时 feature 仍然稳定 | 基础模型迁移 |

这和普通分类预训练的区别很大。分类模型常常把整张图压成一个 `[CLS]` 向量，只要能分对类别就行；DINOv2 更关心的是：**整张 feature map 是否已经组织出视觉世界的结构**。

DINOv2 的工程路线可以从三个层面理解：

1. **数据层面**：从大规模网页图像中做去重、过滤与聚类采样，构造 LVD-142M 这种更干净、更覆盖长尾视觉概念的数据集。
2. **目标层面**：结合图像级自蒸馏、patch 级 masked image modeling、特征分布正则，让模型既学全局语义，也学局部 dense feature。
3. **读出层面**：训练完成后不只读取 `[CLS]`，也读取整张 patch feature map；这让它更接近“视觉基础表征”，而不是单一分类 backbone。

因此课堂上可以把 DINOv2 讲成一句话：

> DINOv2 不是在统一 CV 的输出格式，而是在尝试统一 CV 的中间表征。

论文里也把目标说得很直接：希望模型产生 all-purpose visual features，可以跨图像分布、跨任务使用；官方实现提供的模型从 ViT-S/14 到 ViT-g/14。


In [1]:
# ── DINOv2：提取全局与密集视觉表征 ─────────────────────────────────
# 首次运行会从 torch hub 下载 DINOv2 权重。
# 如果网络不可用，可以先读代码结构：重点看 [CLS] feature 与 patch feature 的区别。
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image, ImageDraw
import requests
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt

# DINOv2 ViT-S/14：patch size = 14，224x224 输入会得到 16x16 = 256 个 patch tokens
# 也可以换成 dinov2_vitb14 / dinov2_vitl14 / dinov2_vitg14
try:
    dinov2_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
    dinov2_model.eval()
    dinov2_ready = True
    print(f"DINOv2 ViT-S/14 参数量: {sum(p.numel() for p in dinov2_model.parameters())/1e6:.1f}M")
except Exception as e:
    dinov2_ready = False
    print("DINOv2 权重暂时无法加载，可能是网络或 torch hub 缓存问题。")
    print("错误信息:", repr(e))

transform_dinov2 = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# 优先使用真实图片；如果课堂现场网络不可用，就退回到一个本地合成示意图。
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content)).convert('RGB')
except Exception as e:
    print("示例图片下载失败，使用本地合成图继续演示。")
    print("错误信息:", repr(e))
    img = Image.new('RGB', (224, 224), color=(235, 235, 230))
    draw = ImageDraw.Draw(img)
    draw.ellipse((42, 48, 178, 184), fill=(210, 160, 96), outline=(80, 60, 40), width=4)
    draw.polygon([(72, 62), (92, 24), (108, 70)], fill=(210, 160, 96), outline=(80, 60, 40))
    draw.polygon([(132, 70), (154, 24), (166, 78)], fill=(210, 160, 96), outline=(80, 60, 40))
    draw.ellipse((86, 104, 98, 116), fill=(20, 20, 20))
    draw.ellipse((126, 104, 138, 116), fill=(20, 20, 20))
    draw.line((112, 122, 104, 138), fill=(80, 60, 40), width=3)
    draw.line((112, 122, 122, 138), fill=(80, 60, 40), width=3)

x = transform_dinov2(img).unsqueeze(0)

if dinov2_ready:
    with torch.no_grad():
        features = dinov2_model.forward_features(x)
        cls_feat = features["x_norm_clstoken"]       # (1, 384): 整图表征
        patch_feats = features["x_norm_patchtokens"] # (1, 256, 384): 16x16 密集 patch 表征

    print(f"[CLS] 全局表征: {cls_feat.shape}")
    print(f"Patch 密集表征: {patch_feats.shape}  (16x16 patches，每个 patch 一个 384 维坐标)")
    print("=> 下游分类可以读 [CLS]；分割、深度、对应、跟踪更关心 patch feature map。")


DINOv2 权重暂时无法加载，可能是网络或 torch hub 缓存问题。
错误信息: <HTTPError 403: 'rate limit exceeded'>
示例图片下载失败，使用本地合成图继续演示。
错误信息: ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='upload.wikimedia.org', port=443): Read timed out. (read timeout=10)"))


### 把 patch feature 看成“视觉坐标系”

DINOv2 最适合在课堂上展示的，不是分类 logits，而是 **patch feature map**。

如果把每个 patch 的高维特征降到 3 维，再映射成 RGB，你会看到一个很有意思的现象：图像里语义相近、结构相连的区域，颜色往往也更接近；物体边界附近，颜色会发生明显变化。

这不是模型被显式要求输出分割图，而是表征空间自己组织出了某种“对象性”。这正好回应前面的问题：CV 的最终任务很碎，但一个好的中间表征可以同时服务很多任务。


In [ ]:
# ── DINOv2 patch feature PCA 可视化 ─────────────────────────────────
# 把 384 维 patch feature 降到 3 维，并映射成 RGB。
# 这不是语义分割标签，而是“表征空间长什么样”的可视化。

def pca_rgb(features):
    """features: (N, D) -> (N, 3), values in [0, 1]."""
    features = features - features.mean(dim=0, keepdim=True)
    # torch.pca_lowrank 避免额外依赖 sklearn
    _, _, v = torch.pca_lowrank(features, q=3)
    projected = features @ v[:, :3]
    projected = (projected - projected.min(dim=0).values) / (
        projected.max(dim=0).values - projected.min(dim=0).values + 1e-6
    )
    return projected

if dinov2_ready:
    patch_grid = int(patch_feats.shape[1] ** 0.5)  # 16
    rgb = pca_rgb(patch_feats[0]).reshape(patch_grid, patch_grid, 3).cpu()
    rgb_up = F.interpolate(
        rgb.permute(2, 0, 1).unsqueeze(0),
        size=(224, 224),
        mode='nearest'
    )[0].permute(1, 2, 0)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img.resize((224, 224)))
    axes[0].set_title('Original image')
    axes[0].axis('off')
    axes[1].imshow(rgb_up.numpy())
    axes[1].set_title('DINOv2 patch features (PCA→RGB)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


### 从“看起来像分割”继续往深处看

DINOv2 的 patch feature 可视化容易让人只记住一句话：“它好像会自动分割”。但更深的理解应该是：

> **分割感只是通用表征的外在表现，不是最终目的。**

好的视觉表征应该像一个可复用的“视觉中间层”：

- 做分类时，读全局 `[CLS]` 或平均 pooled feature；
- 做分割时，读每个 patch 的 dense feature；
- 做检索时，比较图像级向量或局部向量；
- 做深度 / 几何任务时，把 feature map 接到轻量 decoder；
- 做视频对应或点跟踪时，比较不同帧中 patch feature 的相似性。

这也是为什么 DINOv2 在课程里应该接在 CV vs LLM 后面讲：它不是给每个任务重新发明一个模型，而是在逼近一种更基础的东西——**视觉世界的共享表征基底**。

下面的小例子不做分类，也不做分割，只做一件很朴素的事：选一个 patch，看图像中哪些位置和它在特征空间里最像。


In [ ]:
# ── DINOv2 patch 相似度热力图 ──────────────────────────────────────
# 选择一个查询 patch，计算它和所有 patch 的 cosine similarity。
# 这可以帮助学生理解：patch feature 已经携带局部语义/结构信息。

def show_patch_similarity(patch_feats, query_row=8, query_col=8):
    patch_grid = int(patch_feats.shape[1] ** 0.5)
    feats = F.normalize(patch_feats[0], dim=-1)  # (N, D)
    query_idx = query_row * patch_grid + query_col
    query = feats[query_idx:query_idx + 1]
    sim = (query @ feats.T).reshape(patch_grid, patch_grid).cpu()

    sim_up = F.interpolate(
        sim.unsqueeze(0).unsqueeze(0),
        size=(224, 224),
        mode='bilinear',
        align_corners=False,
    )[0, 0]

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img.resize((224, 224)))
    axes[0].scatter(
        [(query_col + 0.5) * 224 / patch_grid],
        [(query_row + 0.5) * 224 / patch_grid],
        c='cyan', s=80, edgecolors='black'
    )
    axes[0].set_title('Query patch')
    axes[0].axis('off')

    axes[1].imshow(img.resize((224, 224)))
    axes[1].imshow(sim_up.numpy(), cmap='magma', alpha=0.65)
    axes[1].set_title('Cosine similarity in feature space')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

if dinov2_ready:
    show_patch_similarity(patch_feats, query_row=8, query_col=8)


---
## Part 3: MAE — 从缺失中学习结构先验

> 论文：*Masked Autoencoders Are Scalable Vision Learners* (He et al., 2021)  
> 链接：<https://arxiv.org/abs/2111.06377>

如果说 DINOv2 更像是在学习“视觉世界的可迁移坐标系”，那 MAE 走的是另一条非常朴素但深刻的路线：

> **把大部分图像盖住，只给模型少量证据，让它恢复被遮住的视觉世界。**

这件事的重点不是“像素重建”本身，而是它迫使 encoder 学到一种结构先验：

- 只看到猫的一部分，要推断身体轮廓和背景关系；
- 只看到道路和天空的局部，要推断场景布局；
- 只看到物体几个角落，要补出整体形状。

现实视觉本来就充满遮挡、局部观察和不完整信息。MAE 把这种不完整性变成训练任务：模型不能只记局部纹理，而要利用上下文、形状、语义和空间布局去做补全。

### 核心设计：高遮掩率 + 非对称 Encoder-Decoder

```
原始图像 (196 patches)
     │  随机遮掩 75%（147 patches）
     ▼
可见 patches（49个） → Encoder（完整 ViT）→ 可见 patch latent
     │  + mask tokens（只在 decoder 里补回）
     ▼
所有 patches → Decoder（轻量 Transformer）→ 重建 masked patch 像素
     │
     ▼
Loss = MSE(重建像素, 原始像素)  仅在 masked patch 上计算
```

| 设计亮点 | 表征角度的解释 |
|---|---|
| **75% 高遮掩率** | 任务足够难，不能靠邻近像素复制，必须理解全局结构 |
| **Encoder 只看可见 patch** | 把算力集中在有效证据上，大幅降低预训练成本 |
| **轻量 Decoder 负责重建** | 让 encoder 更偏向可迁移表征，而不是把所有容量花在像素生成 |
| **只在 masked patch 上算 loss** | 训练信号集中在“从上下文推断缺失内容”这件事上 |

论文里的关键观察是：这种非对称设计不仅让训练更高效，也让大 ViT 更容易扩展；高比例遮掩让任务变得 non-trivial，学到的 encoder 初始化在下游 fine-tuning 中很有价值。


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# ── MAE 随机遮掩策略可视化 ─────────────────────────────────────────
def random_masking(x, mask_ratio=0.75):
    """
    x: (B, N, D)  N 个 patch tokens
    返回: x_masked（可见 patch），mask（1=masked），ids_restore（还原顺序）
    """
    B, N, D = x.shape
    len_keep = int(N * (1 - mask_ratio))

    # 用随机噪声排序来实现随机采样
    noise = torch.rand(B, N)
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1)

    # Encoder 只接收可见 patch
    ids_keep = ids_shuffle[:, :len_keep]
    x_masked = torch.gather(
        x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, D)
    )

    # mask=1 表示该 patch 被遮住；还原到原图顺序，方便 decoder 重建
    mask = torch.ones(B, N)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, 1, ids_restore)

    return x_masked, mask, ids_restore


def show_masking_demo(mask_ratio=0.75, patch_size=16, img_size=224):
    n = img_size // patch_size
    N = n * n
    dummy_patches = torch.zeros(1, N, 1)
    visible, mask, _ = random_masking(dummy_patches, mask_ratio)

    mask_grid = mask[0].reshape(n, n).numpy()
    visible_grid = 1 - mask_grid

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    axes[0].imshow(np.ones((n, n, 3)))
    axes[0].set_title('Full patch grid')
    axes[0].axis('off')

    masked_img = np.ones((n, n, 3))
    masked_img[mask_grid == 1] = [0.18, 0.18, 0.18]
    axes[1].imshow(masked_img)
    axes[1].set_title(f'{mask_ratio*100:.0f}% masked')
    axes[1].axis('off')

    visible_img = np.zeros((n, n, 3))
    visible_img[visible_grid == 1] = [0.95, 0.95, 0.95]
    axes[2].imshow(visible_img)
    axes[2].set_title('Encoder only sees these')
    axes[2].axis('off')

    plt.suptitle('MAE: learn global structure from sparse evidence', fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f"可见 patch 数: {visible.shape[1]} / {N}")
    print("=> Encoder 只看 25% patch，却要支持 decoder 重建被遮住的 75%。")

show_masking_demo(mask_ratio=0.75)


In [ ]:
# ── 使用 MAE encoder 表征：真实工程里通常拿 encoder 去 fine-tune ───────
# 注意：不同 timm 版本对 MAE 权重命名支持不完全一致。
# 如果 pretrained=True 加载失败，可以按官方 repo 下载 MAE checkpoint 后手动加载 encoder 权重。
import timm
import torch

candidate_names = [
    'vit_base_patch16_224.mae',
    'vit_base_patch16_224',
]

mae_model = None
for name in candidate_names:
    try:
        mae_model = timm.create_model(name, pretrained=(name.endswith('.mae')))
        print(f"Loaded model: {name}")
        break
    except Exception as e:
        print(f"Skip {name}: {repr(e)}")

if mae_model is not None:
    mae_model.eval()
    dummy = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        feats = mae_model.forward_features(dummy)
        # timm ViT 通常返回 (B, N+1, D)，其中第 0 个 token 是 [CLS]
        cls_feat = feats[:, 0, :] if feats.ndim == 3 else feats

    print(f"MAE/ViT encoder 输出特征维度: {cls_feat.shape}")
    print("=> MAE 预训练真正被复用的，通常是 encoder；decoder 多数只服务于预训练重建任务。")
else:
    print("未能创建 MAE/ViT 模型，请检查 timm 版本或本地 checkpoint。")


---
## 总结：从 Transformer 结构到通用视觉表征

### 三种方法在本节课中的位置

| | ViT | DINOv2 | MAE |
|---|---|---|---|
| **回答的问题** | 图像怎样进入 Transformer？ | CV 任务碎片化时，能否先统一中间表征？ | 如何从不完整观察中学到结构先验？ |
| **核心对象** | patch token 序列 | 可迁移的全局 + dense patch features | encoder 中的全局结构表征 |
| **训练抓手** | 有监督分类预训练 | 大规模 curated data 上的自蒸馏 / dense feature 学习 | 高比例 masked patch reconstruction |
| **表征特点** | 全局语义强，但依赖任务监督 | 对象性、语义一致性、密集可读性强 | 全局结构、上下文补全、fine-tuning 初始化强 |
| **最适合引出的下游任务** | 分类、检索 | 分割、深度、对应、检索、跨域迁移 | 分类 fine-tuning、检测/分割初始化、生成式预训练思想 |
| **局限** | 只是 token 化和监督分类，不自动解决表征通用性 | 仍然不是一个统一输出头，下游任务还需要 decoder/head | 重建像素不等于直接获得语义，通常依赖 fine-tuning 释放能力 |

### 关键洞见

1. **ViT** 解决的是视觉进入 Transformer 的接口问题：把图像 patch 化，把二维信号变成 token 序列，让 self-attention 可以做全局建模。
2. **DINOv2** 进一步追问：这些 patch token 能否成为可迁移的视觉坐标？它的价值在于学习全局与密集两种表征，让不同 CV 任务都能从同一 feature map 中读取信息。
3. **MAE** 则从“缺失”出发：通过遮住大部分 patch，迫使 encoder 学会利用上下文、形状和场景结构去推断不可见部分，从而形成可迁移的结构先验。
4. **CV 与 LLM 的分野**在这里再次出现：LLM 更容易统一输出 token；CV 很难统一最终任务。因此，视觉基础模型的一条现实路线是先统一中间表征，而不是幻想所有任务共享同一个输出格式。

**发展脉络**：
```
ViT (2020) ──→ MAE (2021) ──→ DINOv2 (2023) ──→ 更通用的视觉基础模型
 图像token化      从缺失学结构        dense visual features       表征基底 + 任务头
```

可以把整节课收成一句话：

> ViT 让图像进入 Transformer；MAE 让模型从不完整观察中学习结构；DINOv2 则把 patch token 推向可迁移、可密集读取的通用视觉表征。它们共同指向一个核心目标：在任务定义高度碎片化的 CV 中，构造一个足够稳定的视觉中间层。


In [ ]:
# ── 表征质量对比：近邻检索（kNN）──────────────────────────────────
# 一个简单但直观的方式：看特征空间中语义近邻是否合理。
# 在真实实验中，可以分别替换为 ViT / DINOv2 / MAE encoder 的输出。
import torch
import torch.nn.functional as F

def knn_retrieval(query_feat, gallery_feats, gallery_labels, k=5):
    """用余弦相似度找 k 个最近邻"""
    q = F.normalize(query_feat, dim=-1)
    g = F.normalize(gallery_feats, dim=-1)
    sims = (q @ g.T).squeeze(0)
    topk_idx = sims.topk(k).indices
    return [(gallery_labels[i], sims[i].item()) for i in topk_idx]

# 模拟：用随机特征演示接口（实际使用时替换为真实模型输出）
torch.manual_seed(42)
D = 768
query = torch.randn(1, D)
gallery = torch.randn(100, D)
labels = [f'class_{i%10}' for i in range(100)]

results = knn_retrieval(query, gallery, labels, k=5)
print("kNN 检索结果（Top-5）:")
for label, sim in results:
    print(f"  {label}  相似度={sim:.4f}")
print("\n=> 好的通用视觉表征，应当让语义/结构相似的样本在特征空间里彼此靠近。")
